In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
## 로드 + split 분리

import os
import math
import random
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from sklearn.metrics import accuracy_score, precision_recall_fscore_support, roc_auc_score

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("DEVICE:", DEVICE)

# 네가 저장한 파일 경로
PT_PATH = "/content/drive/MyDrive/clickbait_data/models_clickbait_cosstats_v1/embeddings_ko_sroberta_multitask_v1.pt"

loaded = torch.load(PT_PATH, map_location="cpu")
samples = loaded["samples"]

train_samples = [x for x in samples if x["split"] == "train"]
valid_samples = [x for x in samples if x["split"] == "valid"]
test_samples  = [x for x in samples if x["split"] == "test"]

print("train:", len(train_samples))
print("valid:", len(valid_samples))
print("test :", len(test_samples))

DEVICE: cpu
train: 481
valid: 199
test : 118


In [ ]:
## Dataset
class ClickbaitEmbeddingDataset(Dataset):
    def __init__(self, samples):
        self.samples = samples

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        x = self.samples[idx]

        item = {
            "id": x["id"],
            "label": torch.tensor(float(x["label_id"]), dtype=torch.float32),

            "title_emb": x["title_emb"].float(),      # (768,)
            "thumb_emb": x["thumb_emb"].float(),      # (768,)
            "stt_embs": x["stt_embs"].float(),        # (Ns, 768)
            "kf_embs": x["kf_embs"].float(),          # (Nk, 768)
        }
        return item

In [ ]:
## collate_fn
def pad_sequence_2d(seq_list, dim=768):
    """
    seq_list: list of tensors [(L1, D), (L2, D), ...]
    return:
      padded: (B, Lmax, D)
      mask:   (B, Lmax)  -> valid=True, pad=False
    """
    batch_size = len(seq_list)
    max_len = max([x.shape[0] for x in seq_list]) if len(seq_list) > 0 else 0

    padded = torch.zeros(batch_size, max_len, dim, dtype=torch.float32)
    mask = torch.zeros(batch_size, max_len, dtype=torch.bool)

    for i, seq in enumerate(seq_list):
        if seq.shape[0] > 0:
            L = seq.shape[0]
            padded[i, :L] = seq
            mask[i, :L] = True

    return padded, mask


def collate_fn(batch):
    title_emb = torch.stack([x["title_emb"] for x in batch], dim=0)   # (B, D)
    thumb_emb = torch.stack([x["thumb_emb"] for x in batch], dim=0)   # (B, D)
    labels = torch.stack([x["label"] for x in batch], dim=0)          # (B,)

    stt_list = [x["stt_embs"] for x in batch]
    kf_list  = [x["kf_embs"] for x in batch]

    stt_padded, stt_mask = pad_sequence_2d(stt_list, dim=title_emb.shape[-1])   # (B, Smax, D), (B, Smax)
    kf_padded, kf_mask   = pad_sequence_2d(kf_list, dim=title_emb.shape[-1])     # (B, Kmax, D), (B, Kmax)

    return {
        "title_emb": title_emb,
        "thumb_emb": thumb_emb,
        "stt_embs": stt_padded,
        "stt_mask": stt_mask,
        "kf_embs": kf_padded,
        "kf_mask": kf_mask,
        "labels": labels,
        "ids": [x["id"] for x in batch]
    }

In [ ]:
## DataLoader
BATCH_SIZE = 32

train_ds = ClickbaitEmbeddingDataset(train_samples)
valid_ds = ClickbaitEmbeddingDataset(valid_samples)
test_ds  = ClickbaitEmbeddingDataset(test_samples)

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True, collate_fn=collate_fn)
valid_loader = DataLoader(valid_ds, batch_size=BATCH_SIZE, shuffle=False, collate_fn=collate_fn)
test_loader  = DataLoader(test_ds, batch_size=BATCH_SIZE, shuffle=False, collate_fn=collate_fn)